# Without english PROMPT and TEST

In [ ]:

!pip -q install transformers accelerate sentencepiece pandas tqdm openpyxl

In [ ]:

import os
import re
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

## 1) GPU

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

True
12.8
1
Tesla T4


## 2) Upload

In [ ]:
from google.colab import files
import pandas as pd

# upload
uploaded = files.upload()

print(list(uploaded.keys()))


filename = list(uploaded.keys())[0]


df = pd.read_csv(filename)

# check
print("\nLoaded file:", filename)
print(df.head())
print("\nColumns:", list(df.columns))
print("\nShape:", df.shape)

Saving dataset_without_english.csv to dataset_without_english (1).csv
['dataset_without_english (1).csv']

Loaded file: dataset_without_english (1).csv
   id                language_A               language_B  \
0   1                   ma daka                  ma daka   
1   2           ma daka du kano          ma daka mu kano   
2   3           ma daka ri toto         ma daka nel toto   
3   4          fila ma vek daka          ma zo daka fila   
4   5  fila ma vek daka du kano  ma zo daka mu kano fila   

             language_C               language_D          language_E  
0                medaka                  daka-ka              madaka  
1       medaka dra kano          daka-ka kano-mu        makanomodaka  
2      medaka vren toto         daka-ka toto-nun        matotonudaka  
3           ardaka fila          daka-ka-ra fila        madakarufila  
4  ardaka dra kano fila  daka-ka-ra kano-mu fila  makanomodakarufila  

Columns: ['id', 'language_A', 'language_B', 'language_C', 'l

In [ ]:

expected_cols = {"id", "language_A", "language_B", "language_C", "language_D", "language_E"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

language_columns = ["language_A", "language_B", "language_C", "language_D", "language_E"]

## 3) QWEN

In [ ]:

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded:", MODEL_NAME)


import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

model = model.to(device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
device = cuda


## 4) Functions

In [ ]:
# function to generate response with Qwen
def generate_with_qwen(system_prompt: str, user_prompt: str, max_new_tokens: int = 300) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## 5) Block

In [ ]:

def build_column_block(df: pd.DataFrame, col: str, max_rows: int | None = None) -> str:
    sub = df[["id", col]].copy()
    if max_rows is not None:
        sub = sub.head(max_rows)

    lines = []
    for _, row in sub.iterrows():
        lines.append(f"{row['id']}\t{row[col]}")
    return "\n".join(lines)

## 6) PROMPT

In [ ]:

SYSTEM_PROMPT_RULES = '''
You are a careful linguist.
You are analyzing ONLY ONE language column from an artificial-language dataset.

CRITICAL RULES:
- Analyze ONLY the forms shown in the user prompt.
- Do NOT mention, compare, or infer anything from other language columns.
- Do NOT invent forms or morphemes not visible in the dataset.
- If uncertain, say "unknown" or "uncertain".
- Keep the answer concise and structured.
- Output ONLY the requested tagged format.

Required output format exactly:

LANGUAGE_COLUMN
<column name>

OBSERVED_UNITS
- <unit> | <possible function> | <short evidence>
- <unit> | <possible function> | <short evidence>

RULES_SUMMARY
<3 to 6 sentences>

CANDIDATE_MARKERS
past: ...
future: ...
present: ...
plural: ...
location_to: ...
location_into: ...
location_in: ...
copula_or_predication: ...
clause_linking: ...
other: ...

TYPOLOGICAL_FEATURES
word_order: ...
subject_marking: ...
object_marking: ...
tense_marking: ...
plural_marking: ...
location_marking: ...
copula_or_predication: ...
clause_linking_or_subordination: ...
fusion_or_boundness: ...
incorporation: ...

UNCERTAINTIES
- ...
- ...
'''.strip()


SYSTEM_PROMPT_TYPE = '''
You are a careful linguist.
You are classifying ONLY ONE language column from an artificial-language dataset.

CRITICAL RULES:
- Analyze ONLY the forms shown in the user prompt.
- Do NOT compare with other columns.
- Do NOT invent hidden grammatical material.
- Base the classification on visible patterns only.
- If uncertain, say so.
- Output ONLY the requested tagged format.

Possible labels:
isolating
analytic
agglutinative
fusional
polysynthetic

Required output format exactly:

LANGUAGE_COLUMN
<column name>

PREDICTED_TYPE
<one label>

CONFIDENCE
<low / medium / high>

REASONING
<3 to 5 sentences>

SUPPORTING_PATTERNS
- <pattern> | <why it matters>
- <pattern> | <why it matters>

ALTERNATIVES
- <alternative 1>
- <alternative 2>

UNCERTAINTIES
- ...
- ...
'''.strip()


SYSTEM_PROMPT_GENERATION = '''
You are a careful linguist.
You are generating new plausible forms for ONLY ONE language column from an artificial-language dataset.

CRITICAL RULES:
- Use ONLY morphemes, markers, and combinations that are already visible in the dataset.
- Do NOT invent new morphemes.
- Do NOT use English.
- Do NOT compare with other columns.
- If you cannot safely generate a form, say "uncertain".
- Output ONLY the requested tagged format.

Required output format exactly:

LANGUAGE_COLUMN
<column name>

GENERATED_FORMS
- <new form> | <reused patterns> | <why plausible>
- <new form> | <reused patterns> | <why plausible>
- <new form> | <reused patterns> | <why plausible>
- <new form> | <reused patterns> | <why plausible>
- <new form> | <reused patterns> | <why plausible>

GENERATION_NOTES
<2 to 4 sentences>

UNCERTAINTIES
- ...
- ...
'''.strip()



SYSTEM_PROMPT_TYPE = """
You are a careful linguist classifying ONLY ONE language column from an artificial-language dataset.

Decide which morphological type is BEST supported by the visible forms in this one column.

STRICT RULES:
- Analyze ONLY the forms shown in the user prompt.
- Do NOT compare with other columns.
- Do NOT invent hidden morphemes or meanings.
- Use only visible repeated formal patterns.
- Do NOT default to agglutinative.
- Use these criteria:
  - isolating: mostly separate free forms, little visible morphology
  - analytic: mostly separate words, grammar expressed by separate elements
  - agglutinative: clear segmentable parts with relatively stable functions
  - fusional: one segment seems to express multiple categories at once
  - polysynthetic: very complex words combining many meaningful elements, possibly including incorporated material
- If evidence is mixed, choose the best label but lower confidence.
- Output ONLY the exact tagged format below.

Required output format exactly:

LANGUAGE_COLUMN
<column name>

PREDICTED_TYPE
<one label>

CONFIDENCE
<low / medium / high>

REASONING
<3 to 5 short sentences based only on visible evidence. Mention one rejected alternative.>

SUPPORTING_PATTERNS
- <pattern> | <why it matters>
- <pattern> | <why it matters>

ALTERNATIVES
- <alternative label> | <why not chosen>
- <alternative label> | <why not chosen>

UNCERTAINTIES
- ...
- ...
""".strip()

In [ ]:
# The other two system prompts are similar in style and can be found above.
def build_rules_prompt(df: pd.DataFrame, col: str, max_rows: int | None = None) -> str:
    block = build_column_block(df, col, max_rows=max_rows)
    return f'''
Analyze the following artificial-language column by itself.

Language column: {col}

Data:
id\tform
{block}

Task:
Infer recurring units, likely markers, and broad grammatical rules for THIS column only.
'''.strip()

def build_type_prompt(df: pd.DataFrame, col: str, max_rows: int | None = None) -> str:
    block = build_column_block(df, col, max_rows=max_rows)
    return f'''
Classify the following artificial-language column by itself.

Language column: {col}

Data:
id\tform
{block}

Task:
Predict the morphological type for THIS column only.
'''.strip()

def build_generation_prompt(df: pd.DataFrame, col: str, max_rows: int | None = None) -> str:
    block = build_column_block(df, col, max_rows=max_rows)
    return f'''
Generate new plausible forms for the following artificial-language column.

Language column: {col}

Data:
id\tform
{block}

Task:
Generate 5 new plausible forms for THIS column only, reusing only visible material.
'''.strip()

## 7) Parsing

In [ ]:
TAG_ORDER_RULES = [
    "LANGUAGE_COLUMN",
    "OBSERVED_UNITS",
    "RULES_SUMMARY",
    "CANDIDATE_MARKERS",
    "TYPOLOGICAL_FEATURES",
    "UNCERTAINTIES",
]

TAG_ORDER_TYPE = [
    "LANGUAGE_COLUMN",
    "PREDICTED_TYPE",
    "CONFIDENCE",
    "REASONING",
    "SUPPORTING_PATTERNS",
    "ALTERNATIVES",
    "UNCERTAINTIES",
]

TAG_ORDER_GENERATION = [
    "LANGUAGE_COLUMN",
    "GENERATED_FORMS",
    "GENERATION_NOTES",
    "UNCERTAINTIES",
]

def parse_tagged_output(text: str, tag_order: list[str]) -> dict:
    result = {tag: "" for tag in tag_order}
    current = None
    buffer = []

    for line in text.splitlines():
        stripped = line.strip()
        matched_tag = None
        for tag in tag_order:
            if stripped == f"==={tag}===":
                matched_tag = tag
                break

        if matched_tag is not None:
            if current is not None:
                result[current] = "\n".join(buffer).strip()
            current = matched_tag
            buffer = []
        else:
            if current is not None:
                buffer.append(line)

    if current is not None:
        result[current] = "\n".join(buffer).strip()

    return result

In [ ]:
# The following helper functions can be used to further process specific sections of the output, such as turning bullet lists into Python lists or parsing key-value blocks into dictionaries.
def bullet_lines_to_list(text: str) -> list[str]:
    out = []
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("- "):
            out.append(s[2:].strip())
    return out

def key_value_block_to_dict(text: str) -> dict:
    out = {}
    for line in text.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            out[k.strip()] = v.strip()
    return out

## 8) Small test

In [ ]:
# For testing and debugging, use a smaller subset of the data
TEST_COL = "language_A"
MAX_ROWS_TEST = 80

In [ ]:
# Build and print the rules prompt and raw output for the test column
prompt_rules_test = build_rules_prompt(df, TEST_COL, max_rows=MAX_ROWS_TEST)
raw_rules_test = generate_with_qwen(
    SYSTEM_PROMPT_RULES,
    prompt_rules_test,
    max_new_tokens=260
)
print(raw_rules_test[:4000])

===LANGUAGE_COLUMN===
language_A

===OBSERVED_UNITS===
- ma | past tense marker | ma daka
- daka | verb root | ma daka
- du | location marker | daka du kano
- ri | location marker | daka ri toto
- vek | auxiliary verb | fila ma vek daka
- naku | auxiliary verb | fila ma vek naku
- tun | auxiliary verb | fila ma tun daka
- nala | location marker | daka nala
- ta | past tense marker | ta daka
- du | location marker | daka du kano
- ri | location marker | daka ri toto
- vek | auxiliary verb | fila ta vek daka
- naku | auxiliary verb | fila ta vek naku
- tun | auxiliary verb | fila ta tun daka
- nala | location marker | daka nala
- lo | past tense marker | lo daka
- du | location marker | daka du kano
- ri | location marker | daka ri toto
- vek | auxiliary verb | fila lo vek daka
- naku | auxiliary verb | fila lo vek naku


In [ ]:
# Parse the raw output into a structured dictionary
parsed_rules_test = parse_tagged_output(raw_rules_test, TAG_ORDER_RULES)
parsed_rules_test

{'LANGUAGE_COLUMN': 'language_A',
 'OBSERVED_UNITS': '- ma | past tense marker | ma daka\n- daka | verb root | ma daka\n- du | location marker | daka du kano\n- ri | location marker | daka ri toto\n- vek | auxiliary verb | fila ma vek daka\n- naku | auxiliary verb | fila ma vek naku\n- tun | auxiliary verb | fila ma tun daka\n- nala | location marker | daka nala\n- ta | past tense marker | ta daka\n- du | location marker | daka du kano\n- ri | location marker | daka ri toto\n- vek | auxiliary verb | fila ta vek daka\n- naku | auxiliary verb | fila ta vek naku\n- tun | auxiliary verb | fila ta tun daka\n- nala | location marker | daka nala\n- lo | past tense marker | lo daka\n- du | location marker | daka du kano\n- ri | location marker | daka ri toto\n- vek | auxiliary verb | fila lo vek daka\n- naku | auxiliary verb | fila lo vek naku',
 'RULES_SUMMARY': '',
 'CANDIDATE_MARKERS': '',
 'TYPOLOGICAL_FEATURES': '',
 'UNCERTAINTIES': ''}

In [ ]:
# Build and print the type prompt and raw output for the test column
prompt_type_test = build_type_prompt(df, TEST_COL, max_rows=MAX_ROWS_TEST)
raw_type_test = generate_with_qwen(
    SYSTEM_PROMPT_TYPE,
    prompt_type_test,
    max_new_tokens=180
)
print(raw_type_test[:3000])

===LANGUAGE_COLUMN===
language_A

===PREDICTED_TYPE===
agglutinative

===CONFIDENCE===
high

===REASONING===
The column shows clear segmentable parts with relatively stable functions, such as "daka" and "naku". These segments can be seen as distinct units that change meaning when combined (e.g., "daka" + "na kano" = "daka na kano"). This pattern aligns with agglutinative morphology.

===SUPPORTING_PATTERNS===
- "daka" | It is a single morpheme that can be combined with different suffixes to form various words.
- "naku" | Similar to "daka", "naku" also appears as a single morpheme that can be combined with suffixes.

===ALTERNATIVES===
- Isolating | The column does


In [ ]:
# Parse the raw output into a structured dictionary
parsed_type_test = parse_tagged_output(raw_type_test, TAG_ORDER_TYPE)
parsed_type_test

{'LANGUAGE_COLUMN': 'language_A',
 'PREDICTED_TYPE': 'agglutinative',
 'CONFIDENCE': 'high',
 'REASONING': 'The column shows clear segmentable parts with relatively stable functions, such as "daka" and "naku". These segments can be seen as distinct units that change meaning when combined (e.g., "daka" + "na kano" = "daka na kano"). This pattern aligns with agglutinative morphology.',
 'SUPPORTING_PATTERNS': '- "daka" | It is a single morpheme that can be combined with different suffixes to form various words.\n- "naku" | Similar to "daka", "naku" also appears as a single morpheme that can be combined with suffixes.',
 'ALTERNATIVES': '- Isolating | The column does',
 'UNCERTAINTIES': ''}

In [ ]:
# Build and print the generation prompt and raw output for the test column
prompt_gen_test = build_generation_prompt(df, TEST_COL, max_rows=MAX_ROWS_TEST)
raw_gen_test = generate_with_qwen(
    SYSTEM_PROMPT_GENERATION,
    prompt_gen_test,
    max_new_tokens=220
)
print(raw_gen_test[:3000])

===LANGUAGE_COLUMN===
language_A

===GENERATED_FORMS===
- ma daka | ma daka | ma daka is a common word in the dataset
- ma daka du kano | ma daka du kano | du (past tense marker) is used with verbs
- ma daka ri toto | ma daka ri toto | ri (indicative mood marker) is used with verbs
- fila ma vek daka | fila ma vek daka | vek (verb prefix) is used with verbs
- fila ma vek daka du kano | fila ma vek daka du kano | du (past tense marker) is used with verbs
- fila ma vek daka ri toto | fila ma vek daka ri toto | ri (indicative mood marker) is used with verbs
- sena ma tun daka | sena ma tun daka | tun (verb suffix) is used with verbs
- sena ma tun daka du kano | sena ma tun daka du


In [ ]:
# Parse the raw output into a structured dictionary
parsed_gen_test = parse_tagged_output(raw_gen_test, TAG_ORDER_GENERATION)
parsed_gen_test

{'LANGUAGE_COLUMN': 'language_A',
 'GENERATED_FORMS': '- ma daka | ma daka | ma daka is a common word in the dataset\n- ma daka du kano | ma daka du kano | du (past tense marker) is used with verbs\n- ma daka ri toto | ma daka ri toto | ri (indicative mood marker) is used with verbs\n- fila ma vek daka | fila ma vek daka | vek (verb prefix) is used with verbs\n- fila ma vek daka du kano | fila ma vek daka du kano | du (past tense marker) is used with verbs\n- fila ma vek daka ri toto | fila ma vek daka ri toto | ri (indicative mood marker) is used with verbs\n- sena ma tun daka | sena ma tun daka | tun (verb suffix) is used with verbs\n- sena ma tun daka du kano | sena ma tun daka du',
 'GENERATION_NOTES': '',
 'UNCERTAINTIES': ''}

## 9) All columns


In [ ]:
MAX_ROWS_PER_COLUMN = 100 

In [ ]:
print("model device:", model.device)
print("param device:", next(model.parameters()).device)

model device: cuda:0
param device: cuda:0


In [ ]:
# Iterate over all language columns and store results in a list of dictionaries
results = []

for col in tqdm(language_columns):
    # Rules
    prompt_rules = build_rules_prompt(df, col, max_rows=MAX_ROWS_PER_COLUMN)
    raw_rules = generate_with_qwen(
        SYSTEM_PROMPT_RULES,
        prompt_rules,
        max_new_tokens=260
    )
    parsed_rules = parse_tagged_output(raw_rules, TAG_ORDER_RULES)

    # Type
    prompt_type = build_type_prompt(df, col, max_rows=MAX_ROWS_PER_COLUMN)
    raw_type = generate_with_qwen(
        SYSTEM_PROMPT_TYPE,
        prompt_type,
        max_new_tokens=180
    )
    parsed_type = parse_tagged_output(raw_type, TAG_ORDER_TYPE)

    # Generation
    prompt_gen = build_generation_prompt(df, col, max_rows=MAX_ROWS_PER_COLUMN)
    raw_gen = generate_with_qwen(
        SYSTEM_PROMPT_GENERATION,
        prompt_gen,
        max_new_tokens=220
    )
    parsed_gen = parse_tagged_output(raw_gen, TAG_ORDER_GENERATION)

    results.append({
        "language_column": col,
        "rules_raw": raw_rules,
        "rules_parsed": parsed_rules,
        "type_raw": raw_type,
        "type_parsed": parsed_type,
        "generation_raw": raw_gen,
        "generation_parsed": parsed_gen,
    })

  0%|          | 0/5 [00:00<?, ?it/s]

## 10) Table

In [ ]:
# Process the results into a DataFrame for easier analysis
rows = []

for r in results:
    pr = r["rules_parsed"]
    pt = r["type_parsed"]
    pg = r["generation_parsed"]

    candidate_markers = key_value_block_to_dict(pr.get("CANDIDATE_MARKERS", ""))
    typological_features = key_value_block_to_dict(pr.get("TYPOLOGICAL_FEATURES", ""))

    row = {
        "language_column": r["language_column"],

        # RULES
        "observed_units": " || ".join(bullet_lines_to_list(pr.get("OBSERVED_UNITS", ""))),
        "rules_summary": pr.get("RULES_SUMMARY", ""),
        "candidate_markers_raw": pr.get("CANDIDATE_MARKERS", ""),
        "uncertainties_rules": " || ".join(bullet_lines_to_list(pr.get("UNCERTAINTIES", ""))),

        # extracted markers
        "past_marker": candidate_markers.get("past", ""),
        "future_marker": candidate_markers.get("future", ""),
        "present_marker": candidate_markers.get("present", ""),
        "plural_marker": candidate_markers.get("plural", ""),
        "location_to_marker": candidate_markers.get("location_to", ""),
        "location_into_marker": candidate_markers.get("location_into", ""),
        "location_in_marker": candidate_markers.get("location_in", ""),
        "copula_or_predication_marker": candidate_markers.get("copula_or_predication", ""),
        "clause_linking_marker": candidate_markers.get("clause_linking", ""),
        "other_markers": candidate_markers.get("other", ""),

        # extracted features
        "word_order": typological_features.get("word_order", ""),
        "subject_marking": typological_features.get("subject_marking", ""),
        "object_marking": typological_features.get("object_marking", ""),
        "tense_marking": typological_features.get("tense_marking", ""),
        "plural_marking": typological_features.get("plural_marking", ""),
        "location_marking": typological_features.get("location_marking", ""),
        "copula_or_predication": typological_features.get("copula_or_predication", ""),
        "clause_linking_or_subordination": typological_features.get("clause_linking_or_subordination", ""),
        "fusion_or_boundness": typological_features.get("fusion_or_boundness", ""),
        "incorporation": typological_features.get("incorporation", ""),

        # TYPE
        "predicted_type": pt.get("PREDICTED_TYPE", ""),
        "confidence": pt.get("CONFIDENCE", ""),
        "reasoning": pt.get("REASONING", ""),
        "supporting_patterns": " || ".join(bullet_lines_to_list(pt.get("SUPPORTING_PATTERNS", ""))),
        "alternatives": " || ".join(bullet_lines_to_list(pt.get("ALTERNATIVES", ""))),
        "uncertainties_type": " || ".join(bullet_lines_to_list(pt.get("UNCERTAINTIES", ""))),

        # GENERATION
        "generated_forms": " || ".join(bullet_lines_to_list(pg.get("GENERATED_FORMS", ""))),
        "generation_notes": pg.get("GENERATION_NOTES", ""),
        "uncertainties_generation": " || ".join(bullet_lines_to_list(pg.get("UNCERTAINTIES", ""))),
    }

    rows.append(row)

results_df = pd.DataFrame(rows)
results_df

,language_column,observed_units,rules_summary,candidate_markers_raw,uncertainties_rules,past_marker,future_marker,present_marker,plural_marker,location_to_marker,...,incorporation,predicted_type,confidence,reasoning,supporting_patterns,alternatives,uncertainties_type,generated_forms,generation_notes,uncertainties_generation
0,language_A,ma | past tense marker | ma daka || daka | ver...,,,,,,,,,...,,agglutinative,high,The forms show clear segmentable parts with re...,"""daka"" | This is a single segment that can sta...",,,ma daka | ma daka | ma daka is a common word i...,The language appears to be a simple agglutinat...,
1,language_B,ma | past tense marker | ma daka || ta | past ...,,,,,,,,,...,,agglutinative,high,The forms show clear segmentable parts with re...,"""daka"" | Represents a noun that can be modifie...",None,Is,ma daka | ma daka | ma daka is a common word i...,The language appears to be a simple agglutinat...,
2,language_C,medaka | noun | id=1 || medaka dra kano | noun...,,,,,,,,,...,,agglutinative,high,The forms show clear segmentable parts with re...,"""dra"" | Indicates a root or base form that can...",Isolating | The forms are mostly separate,,medaka dra kano | medaka dra kano | medaka dra...,,
3,language_D,da-ka | past | daka-ka || da-ka | present | da...,,,,,,,,,...,,agglutinative,high,The column shows a clear pattern of agglutinat...,"""daka"" | The root without any affixes. || ""dak...",,,daka-ka | <reused patterns> | <daka-ka is a co...,,
4,language_E,madaka | noun | || matotonudaka | noun | || ta...,,,,,,,,,...,,agglutinative,high,The forms in this column show clear segmentabl...,madaka | shows a single root word || tadaka | ...,,,madakasesena | matotonudakasesena | matotonuda...,The generated forms are plausible because they...,


In [ ]:
import os
import json

OUT_DIR = "/content/results"
os.makedirs(OUT_DIR, exist_ok=True)

with open(
    os.path.join(
        OUT_DIR,
        "column_annotations_without_english_tagged_raw.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

results_df.to_csv(
    os.path.join(
        OUT_DIR,
        "column_annotations_without_english_tagged_table.csv"
    ),
    index=False
)

results_df.to_excel(
    os.path.join(
        OUT_DIR,
        "column_annotations_without_english_tagged_table.xlsx"
    ),
    index=False
)

print("Saved in:", OUT_DIR)
print(os.listdir(OUT_DIR))

Saved in: /content/results
['column_annotations_without_english_tagged_table.csv', 'column_annotations_without_english_tagged_raw.json', 'column_annotations_without_english_tagged_table.xlsx']


In [ ]:
from google.colab import files

files.download(os.path.join(OUT_DIR, "column_annotations_without_english_tagged_raw.json"))
files.download(os.path.join(OUT_DIR, "column_annotations_without_english_tagged_table.csv"))
files.download(os.path.join(OUT_DIR, "column_annotations_without_english_tagged_table.xlsx"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
print("HOME:", os.path.expanduser("~"))

HOME: /root


In [ ]:

results_df

,language_column,observed_units,rules_summary,candidate_markers_raw,uncertainties_rules,past_marker,future_marker,present_marker,plural_marker,location_to_marker,...,incorporation,predicted_type,confidence,reasoning,supporting_patterns,alternatives,uncertainties_type,generated_forms,generation_notes,uncertainties_generation
0,language_A,ma | past tense marker | ma daka || daka | ver...,,,,,,,,,...,,agglutinative,high,The forms show clear segmentable parts with re...,"""daka"" | This is a single segment that can sta...",,,ma daka | ma daka | ma daka is a common word i...,The language appears to be a simple agglutinat...,
1,language_B,ma | past tense marker | ma daka || ta | past ...,,,,,,,,,...,,agglutinative,high,The forms show clear segmentable parts with re...,"""daka"" | Represents a noun that can be modifie...",None,Is,ma daka | ma daka | ma daka is a common word i...,The language appears to be a simple agglutinat...,
2,language_C,medaka | noun | id=1 || medaka dra kano | noun...,,,,,,,,,...,,agglutinative,high,The forms show clear segmentable parts with re...,"""dra"" | Indicates a root or base form that can...",Isolating | The forms are mostly separate,,medaka dra kano | medaka dra kano | medaka dra...,,
3,language_D,da-ka | past | daka-ka || da-ka | present | da...,,,,,,,,,...,,agglutinative,high,The column shows a clear pattern of agglutinat...,"""daka"" | The root without any affixes. || ""dak...",,,daka-ka | <reused patterns> | <daka-ka is a co...,,
4,language_E,madaka | noun | || matotonudaka | noun | || ta...,,,,,,,,,...,,agglutinative,high,The forms in this column show clear segmentabl...,madaka | shows a single root word || tadaka | ...,,,madakasesena | matotonudakasesena | matotonuda...,The generated forms are plausible because they...,


# THE TESTS

## 1. Ground truth

In [ ]:
import json
import re
import pandas as pd
import os

# 1) LOAD THE RAW RESULTS
RESULTS_PATH = "column_annotations_without_english_tagged_raw (4).json"

with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    llm_results = json.load(f)

print(f"Loaded {len(llm_results)} language results.")


GROUND_TRUTH = {
    "language_A": {
        "true_type": "isolating",
        "true_markers": {
            "future": ["vek"],
            "past": ["tun"],
            "plural": ["nul"],
            "location_to": ["du"],
            "location_into": ["ri"],
            "location_in": ["na"],
            "subject_pronouns": ["ma", "ta", "lo"],
            "copula_or_predication": ["ge"],
            "clause_linking": ["ka"],
            "intensifier": ["po"],
        },
        "true_functions": {
            "vek": ["future"],
            "tun": ["past"],
            "nul": ["plural"],
            "du": ["to", "allative", "destination"],
            "ri": ["into", "illative"],
            "na": ["in", "locative"],
            "ma": ["i", "first person", "subject", "pronoun"],
            "ta": ["you", "second person", "subject", "pronoun"],
            "lo": ["he", "it", "third person", "subject", "pronoun"],
            "ge": ["be", "copula", "predication"],
            "ka": ["because", "clause linking", "subordination"],
            "po": ["very", "intensifier"],
        },
        "true_features": {
            "morphological_type": "isolating",
            "word_structure": "free words",
            "tense_marking": "free particles",
            "plural_marking": "free particle",
            "location_marking": "free particles",
            "affixation": "no affixes",
            "incorporation": False,
        },
    },

    "language_B": {
        "true_type": "analytic",
        "true_markers": {
            "future": ["zo"],
            "past": ["ki"],
            "plural": ["enu"],
            "location_to": ["mu"],
            "location_into": ["nel"],
            "location_in": ["sha"],
            "subject_pronouns": ["ma", "ta", "lo"],
            "copula_or_predication": ["ge"],
            "clause_linking": ["ka"],
            "intensifier": ["po"],
        },
        "true_functions": {
            "zo": ["future", "auxiliary"],
            "ki": ["past", "auxiliary"],
            "enu": ["plural", "classifier"],
            "mu": ["to", "allative", "destination"],
            "nel": ["into", "illative"],
            "sha": ["in", "locative"],
            "ma": ["i", "first person", "subject", "pronoun"],
            "ta": ["you", "second person", "subject", "pronoun"],
            "lo": ["he", "it", "third person", "subject", "pronoun"],
            "ge": ["be", "copula", "predication"],
            "ka": ["because", "clause linking", "subordination"],
            "po": ["very", "intensifier"],
        },
        "true_features": {
            "morphological_type": "analytic",
            "word_structure": "free words",
            "tense_marking": "auxiliaries",
            "plural_marking": "classifier",
            "location_marking": "adpositions",
            "affixation": "low inflection",
            "incorporation": False,
        },
    },

    "language_C": {
        "true_type": "fusional",
        "true_markers": {
            "present_prefixes": ["me", "ti", "na"],
            "past_prefixes": ["uk", "ot", "ev"],
            "future_prefixes": ["ar", "el", "or"],
            "plural_classes": ["esh", "um", "ai", "ok", "ir"],
            "location_to": ["dra"],
            "location_into": ["vren"],
            "location_in": ["nim"],
            "clause_linking": ["ka"],
            "intensifier": ["po"],
        },
        "true_functions": {
            "me": ["present", "first person", "prefix"],
            "ti": ["present", "second person", "prefix"],
            "na": ["present", "third person", "prefix"],
            "uk": ["past", "first person", "prefix"],
            "ot": ["past", "second person", "prefix"],
            "ev": ["past", "third person", "prefix"],
            "ar": ["future", "first person", "prefix"],
            "el": ["future", "second person", "prefix"],
            "or": ["future", "third person", "prefix"],
            "dra": ["to", "allative", "destination"],
            "vren": ["into", "illative"],
            "nim": ["in", "locative"],
            "ka": ["because", "clause linking", "subordination"],
            "po": ["very", "intensifier"],
        },
        "true_features": {
            "morphological_type": "fusional",
            "word_structure": "fused prefix",
            "tense_marking": "person-tense fusion",
            "subject_marking": "fused with tense",
            "plural_marking": "noun class plural",
            "location_marking": "free markers",
            "affixation": "prefixing",
            "incorporation": False,
        },
    },

    "language_D": {
        "true_type": "agglutinative",
        "true_markers": {
            "person": ["ka", "tu", "li"],
            "future": ["ra"],
            "past": ["si"],
            "plural": ["min"],
            "location_to": ["mu"],
            "location_into": ["nun"],
            "location_in": ["ne"],
            "clause_linking": ["ka"],
            "intensifier": ["po"],
        },
        "true_functions": {
            "ka": ["first person", "i", "suffix"],
            "tu": ["second person", "you", "suffix"],
            "li": ["third person", "he", "it", "suffix"],
            "ra": ["future", "suffix"],
            "si": ["past", "suffix"],
            "min": ["plural", "suffix"],
            "mu": ["to", "allative", "case suffix"],
            "nun": ["into", "illative", "case suffix"],
            "ne": ["in", "locative", "case suffix"],
            "po": ["very", "intensifier"],
        },
        "true_features": {
            "morphological_type": "agglutinative",
            "word_structure": "suffix chain",
            "tense_marking": "suffixes",
            "subject_marking": "suffix on verb",
            "plural_marking": "suffix",
            "location_marking": "case suffixes",
            "affixation": "suffixing",
            "incorporation": False,
        },
    },

    "language_E": {
        "true_type": "polysynthetic",
        "true_markers": {
            "subject_prefixes": ["ma", "ta", "lo"],
            "future": ["ru"],
            "past": ["se"],
            "plural": ["it"],
            "location_to": ["mo"],
            "location_into": ["nu"],
            "location_in": ["ni"],
            "clause_linking": ["ka"],
            "intensifier": ["po"],
        },
        "true_functions": {
            "ma": ["first person", "i", "subject prefix"],
            "ta": ["second person", "you", "subject prefix"],
            "lo": ["third person", "he", "it", "subject prefix"],
            "ru": ["future", "suffix"],
            "se": ["past", "suffix"],
            "it": ["plural"],
            "mo": ["to", "allative", "incorporated location"],
            "nu": ["into", "illative", "incorporated location"],
            "ni": ["in", "locative", "incorporated location"],
            "ka": ["because", "clause linking", "subordination"],
            "po": ["very", "intensifier"],
        },
        "true_features": {
            "morphological_type": "polysynthetic",
            "word_structure": "complex predicate",
            "tense_marking": "suffix inside predicate",
            "subject_marking": "subject prefixes",
            "object_marking": "incorporation",
            "location_marking": "incorporation",
            "affixation": "mixed prefixing suffixing",
            "incorporation": True,
        },
    },
}


# DEFINE EVALUATION FUNCTIONS
VALID_TYPES = {
    "isolating",
    "analytic",
    "agglutinative",
    "fusional",
    "polysynthetic",
}

TYPE_ALIASES = {
    "isolating language": "isolating",
    "analytic language": "analytic",
    "agglutinative language": "agglutinative",
    "fusional language": "fusional",
    "fusionnal": "fusional",
    "fusional prefixing": "fusional",
    "polysynthetic language": "polysynthetic",
    "polysynthetic mixed": "polysynthetic",
}

def normalize_text(x):
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return str(x).strip()

def normalize_type(x):
    x = normalize_text(x).lower()
    x = x.replace("morphological type:", "").strip()
    x = x.replace("predicted type:", "").strip()
    x = TYPE_ALIASES.get(x, x)

    for valid in VALID_TYPES:
        if valid in x:
            return valid

    return x

def contains_marker(text, marker):
    text = normalize_text(text).lower()
    marker = normalize_text(marker).lower()

    if not marker:
        return False

    return marker in text


def count_marker_hits(text_blocks, true_markers):
    combined = "\n".join(normalize_text(t) for t in text_blocks).lower()

    details = {}
    hit_count = 0
    total = 0

    for category, markers in true_markers.items():
        category_hit = False
        found_markers = []

        for marker in markers:
            total += 1
            if contains_marker(combined, marker):
                category_hit = True
                hit_count += 1
                found_markers.append(marker)

        details[category] = {
            "expected": markers,
            "found": found_markers,
            "category_hit": int(category_hit),
        }

    score = hit_count / total if total > 0 else None
    return score, hit_count, total, details


def count_function_hits(text_blocks, true_functions):
    combined = "\n".join(normalize_text(t) for t in text_blocks).lower()

    details = {}
    hit_count = 0
    total = 0

    for marker, function_keywords in true_functions.items():
        total += 1

        marker_present = contains_marker(combined, marker)
        matched_keywords = []

        if marker_present:
            for kw in function_keywords:
                if normalize_text(kw).lower() in combined:
                    matched_keywords.append(kw)

        function_hit = int(marker_present and len(matched_keywords) > 0)
        hit_count += function_hit

        details[marker] = {
            "expected_functions": function_keywords,
            "marker_present": int(marker_present),
            "matched_keywords": matched_keywords,
            "function_hit": function_hit,
        }

    score = hit_count / total if total > 0 else None
    return score, hit_count, total, details


def approximate_feature_score(text_blocks, true_features):
    combined = "\n".join(normalize_text(t) for t in text_blocks).lower()

    keyword_map = {
        "isolating": ["isolating", "separate", "free", "particle"],
        "analytic": ["analytic", "auxiliary", "classifier", "free word"],
        "fusional": ["fusional", "fused", "fusion", "single prefix", "person-tense"],
        "agglutinative": ["agglutinative", "suffix", "segmentable", "transparent"],
        "polysynthetic": ["polysynthetic", "incorporation", "complex predicate", "single word"],

        "free words": ["free", "separate", "word"],
        "free particles": ["particle", "free", "separate"],
        "free particle": ["particle", "free", "separate"],
        "auxiliaries": ["auxiliary", "auxiliaries"],
        "classifier": ["classifier"],
        "adpositions": ["adposition", "preposition", "postposition"],

        "fused prefix": ["prefix", "fused", "fusion"],
        "person-tense fusion": ["person", "tense", "fused", "fusion"],
        "fused with tense": ["person", "tense", "fused"],
        "noun class plural": ["plural", "class"],

        "suffix chain": ["suffix", "chain", "agglutinative", "segmentable"],
        "suffixes": ["suffix", "suffixes"],
        "suffix on verb": ["suffix", "verb"],
        "case suffixes": ["case", "suffix"],
        "suffixing": ["suffix", "suffixing"],

        "complex predicate": ["complex", "predicate", "single word"],
        "suffix inside predicate": ["suffix", "predicate"],
        "subject prefixes": ["subject", "prefix"],
        "incorporation": ["incorporation", "incorporated", "complex word"],
        "mixed prefixing suffixing": ["prefix", "suffix", "mixed"],

        "no affixes": ["no affix", "no prefixes", "no suffixes", "no affixes"],
        "low inflection": ["low inflection", "little morphology", "low morphology"],
        "prefixing": ["prefix", "prefixing"],
    }

    details = {}
    hit_count = 0
    total = 0

    for feat_name, feat_value in true_features.items():
        total += 1

        if isinstance(feat_value, bool):
            expected_keywords = (
                ["incorporation", "incorporated", "complex predicate"]
                if feat_value
                else ["no incorporation", "not incorporated", "separate", "free"]
            )
        else:
            expected_keywords = keyword_map.get(
                str(feat_value).lower(),
                [str(feat_value).lower()]
            )

        matched = any(k in combined for k in expected_keywords)
        hit_count += int(matched)

        details[feat_name] = {
            "expected": feat_value,
            "keywords_checked": expected_keywords,
            "matched": int(matched),
        }

    score = hit_count / total if total > 0 else None
    return score, hit_count, total, details


def get_text_blocks(item):
    if "rules_parsed" in item and isinstance(item["rules_parsed"], dict):
        rules_text = "\n".join(
            f"{k}: {v}" for k, v in item["rules_parsed"].items()
        )
    else:
        rules_text = normalize_text(item.get("rules_raw", ""))

    if "type_parsed" in item and isinstance(item["type_parsed"], dict):
        type_text = "\n".join(
            f"{k}: {v}" for k, v in item["type_parsed"].items()
        )
    else:
        type_text = normalize_text(item.get("type_raw", ""))

    if "generation_parsed" in item and isinstance(item["generation_parsed"], dict):
        generation_text = "\n".join(
            f"{k}: {v}" for k, v in item["generation_parsed"].items()
        )
    else:
        generation_text = normalize_text(item.get("generation_raw", ""))

    return rules_text, type_text, generation_text


# extract generated forms into a structured list of dictionaries with "form" and "meaning" keys, trying both the parsed and raw sections
def extract_generated_forms(item):
    forms = []

    gp = item.get("generation_parsed", {})

    if isinstance(gp, dict):
        if "generated_items" in gp and isinstance(gp["generated_items"], list):
            for x in gp["generated_items"]:
                if isinstance(x, dict):
                    form = normalize_text(
                        x.get("predicted_form_unknown_language", "")
                    )
                    meaning = normalize_text(x.get("target_meaning_en", ""))
                    if form:
                        forms.append({"meaning": meaning, "form": form})
        else:
            generated_text = normalize_text(gp.get("GENERATED_FORMS", ""))

            if generated_text:
                for line in generated_text.splitlines():
                    line = line.strip().lstrip("-").strip()
                    if "|" in line:
                        parts = [p.strip() for p in line.split("|")]
                        if len(parts) >= 2:
                            forms.append({
                                "form": parts[0],
                                "meaning": parts[-1],
                            })
                    elif line:
                        forms.append({
                            "form": line,
                            "meaning": "",
                        })

    if not forms:
        raw = normalize_text(item.get("generation_raw", ""))

        for line in raw.splitlines():
            line = line.strip().lstrip("-").strip()
            if "|" in line:
                parts = [p.strip() for p in line.split("|")]
                if len(parts) >= 2:
                    forms.append({
                        "form": parts[0],
                        "meaning": parts[-1],
                    })

    return forms


# score the generated forms based on the presence of expected markers for certain meanings, according to the ground truth for each language
def score_generated_forms(lang, generated_forms):
    if not generated_forms:
        return None, 0, 0, {}

    hit_count = 0
    total = 0
    details = {}

    for i, item in enumerate(generated_forms):
        meaning = normalize_text(item.get("meaning", "")).lower()
        form = normalize_text(item.get("form", "")).lower()

        checks = []

        if lang == "language_A":
            tokens = form.split()

            if "will" in meaning or "tomorrow" in meaning:
                checks.append(("future_vek", "vek" in tokens))

            if "went" in meaning or "yesterday" in meaning:
                checks.append(("past_tun", "tun" in tokens))

            if "to the city" in meaning:
                checks.append(("location_to_du", "du" in tokens))

            if "into the house" in meaning:
                checks.append(("location_into_ri", "ri" in tokens))

            if "in the city" in meaning or "in the house" in meaning:
                if "into" not in meaning:
                    checks.append(("location_in_na", "na" in tokens))

        elif lang == "language_B":
            tokens = form.split()

            if "will" in meaning or "tomorrow" in meaning:
                checks.append(("future_zo", "zo" in tokens))

            if "went" in meaning or "yesterday" in meaning:
                checks.append(("past_ki", "ki" in tokens))

            if "to the city" in meaning:
                checks.append(("location_to_mu", "mu" in tokens))

            if "into the house" in meaning:
                checks.append(("location_into_nel", "nel" in tokens))

            if "in the city" in meaning or "in the house" in meaning:
                if "into" not in meaning:
                    checks.append(("location_in_sha", "sha" in tokens))

        elif lang == "language_C":
            if "will" in meaning or "tomorrow" in meaning:
                checks.append(("future_prefix", any(x in form for x in ["ar", "el", "or"])))

            if "went" in meaning or "yesterday" in meaning:
                checks.append(("past_prefix", any(x in form for x in ["uk", "ot", "ev"])))

            if "to the city" in meaning:
                checks.append(("location_to_dra", "dra" in form))

            if "into the house" in meaning:
                checks.append(("location_into_vren", "vren" in form))

            if "in the city" in meaning or "in the house" in meaning:
                if "into" not in meaning:
                    checks.append(("location_in_nim", "nim" in form))

        elif lang == "language_D":
            if "will" in meaning or "tomorrow" in meaning:
                checks.append(("future_ra", "ra" in form))

            if "went" in meaning or "yesterday" in meaning:
                checks.append(("past_si", "si" in form))

            if "to the city" in meaning:
                checks.append(("location_to_mu", "mu" in form))

            if "into the house" in meaning:
                checks.append(("location_into_nun", "nun" in form))

            if "in the city" in meaning or "in the house" in meaning:
                if "into" not in meaning:
                    checks.append(("location_in_ne", "ne" in form))

        elif lang == "language_E":
            if "will" in meaning or "tomorrow" in meaning:
                checks.append(("future_ru", "ru" in form))

            if "went" in meaning or "yesterday" in meaning:
                checks.append(("past_se", "se" in form))

            if meaning.startswith("i "):
                checks.append(("subject_ma", form.startswith("ma")))

            if meaning.startswith("you "):
                checks.append(("subject_ta", form.startswith("ta")))

            if meaning.startswith("he ") or meaning.startswith("it "):
                checks.append(("subject_lo", form.startswith("lo")))

            if "to the city" in meaning:
                checks.append(("location_to_mo", "mo" in form))

            if "into the house" in meaning:
                checks.append(("location_into_nu", "nu" in form))

            if "in the city" in meaning or "in the house" in meaning:
                if "into" not in meaning:
                    checks.append(("location_in_ni", "ni" in form))

        valid = all(v for _, v in checks) if checks else False

        total += 1
        hit_count += int(valid)

        details[f"item_{i+1}"] = {
            "meaning": meaning,
            "form": form,
            "checks": {k: int(v) for k, v in checks},
            "valid": int(valid),
        }

    score = hit_count / total if total > 0 else None
    return score, hit_count, total, details


# run the evaluation for each language and compile results into a summary DataFrame, as well as a detailed dictionary for each language
rows = []
full_details = {}

for item in llm_results:
    lang = item["language_column"]
    truth = GROUND_TRUTH[lang]

    rules_text, type_text, generation_text = get_text_blocks(item)

    if "type_parsed" in item and isinstance(item["type_parsed"], dict):
        predicted_type = normalize_type(item["type_parsed"].get("PREDICTED_TYPE", ""))
    else:
        m = re.search(r"===PREDICTED_TYPE===\s*(.+)", type_text, flags=re.I)
        predicted_type = normalize_type(m.group(1)) if m else ""

    true_type = truth["true_type"]
    type_correct = int(predicted_type == true_type)

    marker_score, marker_hits, marker_total, marker_details = count_marker_hits(
        [rules_text, type_text, generation_text],
        truth["true_markers"],
    )

    function_score, function_hits, function_total, function_details = count_function_hits(
        [rules_text, type_text],
        truth["true_functions"],
    )

    feature_score, feature_hits, feature_total, feature_details = approximate_feature_score(
        [rules_text, type_text],
        truth["true_features"],
    )

    generated_forms = extract_generated_forms(item)

    generation_score, generation_hits, generation_total, generation_details = score_generated_forms(
        lang,
        generated_forms,
    )

    overall_score = round(
        (
            type_correct
            + (marker_score if marker_score is not None else 0)
            + (function_score if function_score is not None else 0)
            + (feature_score if feature_score is not None else 0)
            + (generation_score if generation_score is not None else 0)
        ) / 5,
        3,
    )

    rows.append({
        "language_column": lang,
        "true_type": true_type,
        "predicted_type": predicted_type,
        "type_correct": type_correct,

        "marker_hits": marker_hits,
        "marker_total": marker_total,
        "marker_score": round(marker_score, 3) if marker_score is not None else None,

        "function_hits": function_hits,
        "function_total": function_total,
        "function_score": round(function_score, 3) if function_score is not None else None,

        "feature_hits": feature_hits,
        "feature_total": feature_total,
        "feature_score": round(feature_score, 3) if feature_score is not None else None,

        "generation_hits": generation_hits,
        "generation_total": generation_total,
        "generation_score": round(generation_score, 3) if generation_score is not None else None,

        "overall_score": overall_score,
    })

    full_details[lang] = {
        "true_type": true_type,
        "predicted_type": predicted_type,
        "type_correct": type_correct,
        "marker_details": marker_details,
        "function_details": function_details,
        "feature_details": feature_details,
        "generation_details": generation_details,
        "generated_forms": generated_forms,
        "rules_text": rules_text,
        "type_text": type_text,
        "generation_text": generation_text,
    }

comparison_df = (
    pd.DataFrame(rows)
    .sort_values("language_column")
    .reset_index(drop=True)
)


# save
comparison_df.to_csv("comparison_without_english.csv", index=False)

with pd.ExcelWriter("comparison_without_english.xlsx", engine="openpyxl") as writer:
    comparison_df.to_excel(writer, index=False, sheet_name="summary")

    for lang, details in full_details.items():

        marker_rows = []
        for cat, d in details["marker_details"].items():
            marker_rows.append({
                "category": cat,
                "expected_markers": ", ".join(d["expected"]),
                "found_markers": ", ".join(d["found"]),
                "category_hit": d["category_hit"],
            })

        pd.DataFrame(marker_rows).to_excel(
            writer,
            index=False,
            sheet_name=f"{lang}_markers"[:31],
        )

        function_rows = []
        for marker, d in details["function_details"].items():
            function_rows.append({
                "marker": marker,
                "expected_functions": ", ".join(d["expected_functions"]),
                "marker_present": d["marker_present"],
                "matched_keywords": ", ".join(d["matched_keywords"]),
                "function_hit": d["function_hit"],
            })

        pd.DataFrame(function_rows).to_excel(
            writer,
            index=False,
            sheet_name=f"{lang}_functions"[:31],
        )

        generation_rows = []
        for gen_id, d in details["generation_details"].items():
            generation_rows.append({
                "item": gen_id,
                "meaning": d["meaning"],
                "form": d["form"],
                "valid": d["valid"],
                "checks": json.dumps(d["checks"], ensure_ascii=False),
            })

        pd.DataFrame(generation_rows).to_excel(
            writer,
            index=False,
            sheet_name=f"{lang}_generation"[:31],
        )

with open("comparison_without_english_details.json", "w", encoding="utf-8") as f:
    json.dump(full_details, f, ensure_ascii=False, indent=2)


# results summary
print("Saved:")
print("- comparison_without_english.csv")
print("- comparison_without_english.xlsx")
print("- comparison_without_english_details.json")

print("\n=== SUMMARY TABLE ===")
print(comparison_df)

print("\n=== AGGREGATE SCORES ===")
print(f"Type accuracy: {comparison_df['type_correct'].mean():.3f}")
print(f"Mean marker score: {comparison_df['marker_score'].mean():.3f}")
print(f"Mean function score: {comparison_df['function_score'].mean():.3f}")
print(f"Mean feature score: {comparison_df['feature_score'].mean():.3f}")
print(f"Mean generation score: {comparison_df['generation_score'].mean():.3f}")
print(f"Mean overall score: {comparison_df['overall_score'].mean():.3f}")

print("\nCurrent working directory:")
print(os.getcwd())

print("\nFiles in directory:")
print(os.listdir())